In [1]:
import os
import cv2
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

ModuleNotFoundError: No module named 'torchvision'

In [ ]:
# ------------------------
# Dataset PyTorch pour eyetracking
# ------------------------
class EyeTrackingDataset(Dataset):
    def __init__(self, csv_file, img_dir, img_size=(224,224), transform=None):
        """
        csv_file : chemin vers le csv (frame_id, timestamp, x, y)
        img_dir : dossier contenant les images (frame_XXXX.png)
        img_size : taille pour redimensionner les images
        """
        self.df = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.img_size = img_size
        self.transform = transform

        # créer mapping frame_id -> image path
        self.img_paths = [os.path.join(img_dir, f"frame_{int(fid)}.png") 
                          for fid in self.df['frame_id']]

        # extraire les coordonnées
        self.coords = self.df[['x','y']].values.astype(np.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # resize
        img = cv2.resize(img, self.img_size)

        # normalisation coords
        h, w, _ = img.shape
        x, y = self.coords[idx]
        x_norm = x / w
        y_norm = y / h
        label = torch.tensor([x_norm, y_norm], dtype=torch.float32)

        # transform PyTorch (ToTensor + normalisation)
        if self.transform:
            img = self.transform(img)
        else:
            # default transform
            img = transforms.ToTensor()(img)
            img = transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])(img)

        return img, label


In [ ]:
# ------------------------
# Exemple d'utilisation
# ------------------------
csv_file = "coords.csv"
img_dir = "frames"

dataset = EyeTrackingDataset(csv_file, img_dir)

# split train/dev/test
from sklearn.model_selection import train_test_split
indices = np.arange(len(dataset))
train_idx, test_idx = train_test_split(indices, test_size=0.3, random_state=42)
dev_idx, test_idx = train_test_split(test_idx, test_size=0.5, random_state=42)

from torch.utils.data import Subset

train_ds = Subset(dataset, train_idx)
dev_ds   = Subset(dataset, dev_idx)
test_ds  = Subset(dataset, test_idx)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
dev_loader   = DataLoader(dev_ds, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False)

# ------------------------
# Vérification rapide
# ------------------------
imgs, labels = next(iter(train_loader))
print(imgs.shape, labels.shape)  # imgs: [B,3,H,W], labels: [B,2]
